# ISIC 2019 setup: tables and folders

This notebook creates the ISIC 2019 dataset-specific Bronze tables and (later) Gold tables, plus their storage folders. The shared Silver tables are created once in `notebooks/00_setup_storage_and_shared_tables.ipynb`, not here.

Run this after `notebooks/00_setup_storage_and_shared_tables.ipynb` and before `10_bronze_ingest.ipynb`. It also prints the exact landing paths where the source archives must exist.


In [ ]:
%run ../_setup_env


In [ ]:
from data_platform.dataset_layout import build_layout, load_dataset_config, load_yaml_config, resolve_archive_paths

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "bronze" / "datasets" / "isic_2019.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]

ARCHIVES = resolve_archive_paths(DATASET_CONFIG["archives"], landing_paths, bronze_paths)

print(layout)
for archive in ARCHIVES:
    print(f"{archive['source_split']} archive upload path: {archive['archive_dbfs_path']}")
    print(f"{archive['source_split']} Bronze image index table: {bronze_tables['image_index']}")
    print(f"{archive['source_split']} Bronze metadata path: {archive['metadata_target_path']}")


## Create Bronze Objects

Create the ISIC 2019 Bronze source metadata table, image index table (checksums and archive locators only — no image bytes, see `docs/decisions/006-stream-archives-no-blob-storage.md`), and dataset-specific landing and metadata folders.


In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.isic_2019_image_index (
      image_id STRING,
      source_split STRING,
      archive_member_path STRING,
      source_archive_uri STRING,
      byte_length BIGINT,
      source_checksum STRING
    )
    USING DELTA
    """
)
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.isic_2019_source_metadata (
      image_id STRING,
      source_split STRING,
      source_uri STRING,
      attribution STRING,
      copyright_license STRING,
      age_approx STRING,
      anatom_site_1 STRING,
      anatom_site_2 STRING,
      anatom_site_3 STRING,
      anatom_site_4 STRING,
      anatom_site_5 STRING,
      anatom_site_special STRING,
      clin_size_long_diam_mm STRING,
      concomitant_biopsy STRING,
      dermoscopic_type STRING,
      diagnosis_1 STRING,
      diagnosis_2 STRING,
      diagnosis_3 STRING,
      diagnosis_4 STRING,
      diagnosis_5 STRING,
      diagnosis_confirm_type STRING,
      family_hx_mm STRING,
      image_type STRING,
      lesion_id STRING,
      melanocytic STRING,
      patient_id STRING,
      personal_hx_mm STRING,
      sex STRING,
      source_checksum STRING,
      ingestion_run_id STRING,
      ingested_at TIMESTAMP
    )
    USING DELTA
    """
)
dbutils.fs.mkdirs(landing_paths["dataset_archive"])
for path in bronze_paths.values():
    dbutils.fs.mkdirs(path)
for archive in ARCHIVES:
    dbutils.fs.mkdirs(archive["metadata_dir_path"])

display(spark.sql("SHOW TABLES IN bronze"))
display(dbutils.fs.ls(landing_paths["archives"]))
display(dbutils.fs.ls(bronze_paths["root"]))


## Gold

`gold.manifest_rows` and `gold.manifest_registry` are shared, not dataset-specific — created once in `notebooks/00_setup_storage_and_shared_tables.ipynb`, same reasoning as Silver. Nothing to do in this notebook.


## Upload Checklist

Upload each archive listed in `config/bronze/datasets/isic_2019.yaml` to the corresponding path printed above before running `10_bronze_ingest.ipynb`.
